# 深度学习实训项目 - 神经网络后向传播的实现

**课程**: 深度学习基础与应用  
**学校**: 广东工商职业技术大学  
**学期**: 2025-2026学年第2学期  

---

## 环境准备

导入必要的库

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"TensorFlow版本: {tf.__version__}")
print(f"NumPy版本: {np.__version__}")

---

# 任务一：使用Keras高层API搭建神经网络实现手写数字分类

## 1.1 加载并预处理MNIST数据集

In [ ]:
def load_and_preprocess_data():
    """
    加载MNIST数据集并进行预处理
    
    Returns:
        tuple: (x_train, y_train, x_test, y_test) 预处理后的数据集
    """
    # 加载MNIST数据集
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    
    # 归一化：将像素值从 [0, 255] 缩放到 [0, 1]
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    
    # 将图像从 28x28 展平为 784 维向量
    x_train = x_train.reshape((x_train.shape[0], 28 * 28))
    x_test = x_test.reshape((x_test.shape[0], 28 * 28))
    
    # 将标签转换为one-hot编码
    y_train = to_categorical(y_train, 10)
    y_test = to_categorical(y_test, 10)
    
    print(f"训练集形状: {x_train.shape}, 标签形状: {y_train.shape}")
    print(f"测试集形状: {x_test.shape}, 标签形状: {y_test.shape}")
    
    return x_train, y_train, x_test, y_test

# 加载数据
x_train, y_train, x_test, y_test = load_and_preprocess_data()

## 1.2 可视化样本数据

In [ ]:
# 可视化部分训练样本
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.title(f'标签: {np.argmax(y_train[i])}')
    plt.axis('off')
plt.suptitle('MNIST训练样本示例')
plt.tight_layout()
plt.show()

## 1.3 构建神经网络模型

In [ ]:
def build_model():
    """
    使用Keras Sequential API构建神经网络模型
    
    模型结构：
    - 输入层：784个神经元（对应28x28图像展平）
    - 隐藏层1：256个神经元，ReLU激活
    - Dropout层：防止过拟合
    - 隐藏层2：128个神经元，ReLU激活
    - 输出层：10个神经元，Softmax激活（对应10个数字类别）
    
    Returns:
        keras.Model: 编译好的神经网络模型
    """
    model = models.Sequential([
        # 输入层
        layers.Input(shape=(784,)),
        
        # 第一隐藏层
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        
        # 第二隐藏层
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        
        # 输出层
        layers.Dense(10, activation='softmax')
    ])
    
    # 编译模型
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("模型结构：")
    model.summary()
    
    return model

# 构建模型
model = build_model()

## 1.4 前向传播与反向传播演示

In [ ]:
def demonstrate_forward_backward_propagation(model, x_train, y_train):
    """
    演示前向传播和反向传播过程
    """
    print("\n" + "="*60)
    print("前向传播与反向传播演示")
    print("="*60)
    
    # 取一个样本
    sample_idx = 0
    x_sample = x_train[sample_idx:sample_idx+1]
    y_sample = y_train[sample_idx:sample_idx+1]
    
    print(f"\n选取样本索引: {sample_idx}")
    print(f"输入形状: {x_sample.shape}")
    print(f"标签形状: {y_sample.shape}")
    print(f"真实标签: {np.argmax(y_sample)}")
    
    # 前向传播：获取各层输出
    print("\n--- 前向传播过程 ---")
    
    # 先进行一次预测，建立模型连接
    _ = model.predict(x_sample, verbose=0)
    
    # 创建逐层输出模型
    from tensorflow.keras.layers import Input
    input_layer = Input(shape=(784,))
    layer_outputs = []
    x = input_layer
    for layer in model.layers:
        x = layer(x)
        layer_outputs.append(x)
    activation_model = models.Model(inputs=input_layer, outputs=layer_outputs)
    
    activations = activation_model.predict(x_sample, verbose=0)
    
    for i, (layer, activation) in enumerate(zip(model.layers, activations)):
        print(f"层 {i+1} ({layer.name}): 输出形状 = {activation.shape}")
        if i == len(model.layers) - 1:
            print(f"  输出概率分布: {activation[0].round(4)}")
            print(f"  预测类别: {np.argmax(activation[0])}")
    
    # 反向传播：计算梯度
    print("\n--- 反向传播过程（梯度计算） ---")
    
    x_tensor = tf.convert_to_tensor(x_sample, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y_sample, dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        predictions = model(x_tensor)
        loss = tf.keras.losses.categorical_crossentropy(y_tensor, predictions)
    
    # 获取可训练变量的梯度
    gradients = tape.gradient(loss, model.trainable_variables)
    
    print(f"损失值: {loss.numpy()[0]:.6f}")
    print("\n各层参数梯度统计:")
    
    for i, (grad, var) in enumerate(zip(gradients, model.trainable_variables)):
        if grad is not None:
            print(f"  参数 {i+1} ({var.name}):")
            print(f"    形状: {grad.shape}")
            print(f"    梯度均值: {tf.reduce_mean(tf.abs(grad)).numpy():.6f}")
            print(f"    梯度最大值: {tf.reduce_max(tf.abs(grad)).numpy():.6f}")
    
    print("\n" + "="*60)

# 运行演示
demonstrate_forward_backward_propagation(model, x_train, y_train)

## 1.5 训练模型

In [ ]:
# 训练模型
print("\n开始训练模型...")
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

## 1.6 评估模型

In [ ]:
# 评估模型
print("\n评估模型性能...")
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"测试集损失: {test_loss:.4f}")
print(f"测试集准确率: {test_accuracy:.4f}")

## 1.7 保存和加载模型

In [ ]:
# 保存模型
model.save('mnist_model.keras')
print("模型已保存到: mnist_model.keras")

# 加载模型
loaded_model = keras.models.load_model('mnist_model.keras')
print("模型已加载")

# 验证加载的模型
loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print(f"加载模型测试准确率: {loaded_acc:.4f}")

## 1.8 可视化训练历史

In [ ]:
# 绘制训练历史
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot(history.history['loss'], label='训练损失')
axes[0].plot(history.history['val_loss'], label='验证损失')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练与验证损失')
axes[0].legend()
axes[0].grid(True)

# 准确率曲线
axes[1].plot(history.history['accuracy'], label='训练准确率')
axes[1].plot(history.history['val_accuracy'], label='验证准确率')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('训练与验证准确率')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('task1_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("训练历史曲线已保存")

## 1.9 可视化预测结果

In [ ]:
# 可视化预测结果
num_samples = 10
indices = np.random.choice(len(x_test), num_samples, replace=False)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    sample = x_test[idx:idx+1]
    prediction = model.predict(sample, verbose=0)
    predicted_label = np.argmax(prediction)
    true_label = np.argmax(y_test[idx])
    
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    color = 'green' if predicted_label == true_label else 'red'
    plt.title(f'预测: {predicted_label}\n真实: {true_label}', color=color)
    plt.axis('off')

plt.tight_layout()
plt.savefig('task1_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("预测可视化已保存")

---

# 任务二：对比交叉熵与均方误差损失对手写数字分类的影响

## 2.1 损失函数特性分析

In [ ]:
print("""
【交叉熵损失 (Categorical Crossentropy)】
1. 数学公式: L = -Σ(y_true * log(y_pred))
2. 特点:
   - 专门针对分类任务设计
   - 对预测概率与真实标签的差异敏感
   - 梯度在误差大时较大，收敛速度快
   - 与Softmax激活函数配合效果最佳
3. 适用场景: 多分类任务的首选损失函数

【均方误差损失 (Mean Squared Error)】
1. 数学公式: L = (1/n) * Σ(y_true - y_pred)²
2. 特点:
   - 通用损失函数，originally用于回归任务
   - 梯度随误差线性变化
   - 在分类任务中可能导致梯度消失问题
   - 对离群值敏感
3. 适用场景: 回归任务，不推荐用于分类任务

【关键差异】
1. 梯度特性:
   - 交叉熵: 梯度与误差成正比，学习信号强
   - MSE: 梯度受Softmax导数影响，可能过小

2. 收敛速度:
   - 交叉熵: 收敛更快，通常需要更少epoch
   - MSE: 收敛较慢，可能需要更多epoch

3. 最终性能:
   - 交叉熵: 在分类任务上通常达到更高准确率
   - MSE: 在分类任务上性能通常较差
""")

## 2.2 使用交叉熵损失训练模型

In [ ]:
# 构建模型
model_ce = models.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# 编译模型（交叉熵损失）
model_ce.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n使用交叉熵损失训练模型...")
history_ce = model_ce.fit(
    x_train, y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

# 评估
loss_ce, acc_ce = model_ce.evaluate(x_test, y_test, verbose=0)
print(f"\n交叉熵 - 测试损失: {loss_ce:.4f}, 测试准确率: {acc_ce:.4f}")

## 2.3 使用均方误差损失训练模型

In [ ]:
# 构建模型
model_mse = models.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# 编译模型（MSE损失）
model_mse.compile(
    optimizer='adam',
    loss='mse',
    metrics=['accuracy']
)

print("\n使用均方误差损失训练模型...")
history_mse = model_mse.fit(
    x_train, y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

# 评估
loss_mse, acc_mse = model_mse.evaluate(x_test, y_test, verbose=0)
print(f"\nMSE - 测试损失: {loss_mse:.4f}, 测试准确率: {acc_mse:.4f}")

## 2.4 对比训练结果

In [ ]:
print("\n" + "="*60)
print("训练结果对比")
print("="*60)

print("\n【最终训练结果对比】")
print(f"{'指标':<25} {'交叉熵':<15} {'均方误差':<15}")
print("-" * 60)
print(f"{'最终训练损失':<25} {history_ce.history['loss'][-1]:<15.4f} {history_mse.history['loss'][-1]:<15.4f}")
print(f"{'最终训练准确率':<25} {history_ce.history['accuracy'][-1]:<15.4f} {history_mse.history['accuracy'][-1]:<15.4f}")
print(f"{'最终验证损失':<25} {history_ce.history['val_loss'][-1]:<15.4f} {history_mse.history['val_loss'][-1]:<15.4f}")
print(f"{'最终验证准确率':<25} {history_ce.history['val_accuracy'][-1]:<15.4f} {history_mse.history['val_accuracy'][-1]:<15.4f}")

print("\n【测试集结果】")
print(f"交叉熵 - 损失: {loss_ce:.4f}, 准确率: {acc_ce:.4f}")
print(f"MSE    - 损失: {loss_mse:.4f}, 准确率: {acc_mse:.4f}")

## 2.5 梯度更新示例

In [ ]:
def demonstrate_gradient_update(model, loss_name, x_train, y_train):
    """
    演示梯度更新过程
    """
    print(f"\n{'='*60}")
    print(f"梯度更新示例 - {loss_name}")
    print(f"{'='*60}")
    
    sample_idx = 0
    x_sample = x_train[sample_idx:sample_idx+1]
    y_sample = y_train[sample_idx:sample_idx+1]
    
    x_tensor = tf.convert_to_tensor(x_sample, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y_sample, dtype=tf.float32)
    
    if loss_name == 'categorical_crossentropy':
        loss_fn = tf.keras.losses.CategoricalCrossentropy()
    else:
        loss_fn = tf.keras.losses.MeanSquaredError()
    
    with tf.GradientTape() as tape:
        predictions = model(x_tensor)
        loss = loss_fn(y_tensor, predictions)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    
    print(f"\n样本索引: {sample_idx}")
    print(f"真实标签: {np.argmax(y_sample)}")
    print(f"预测概率: {predictions.numpy()[0].round(4)}")
    print(f"预测类别: {np.argmax(predictions.numpy()[0])}")
    print(f"\n损失值 ({loss_name}): {loss.numpy():.6f}")
    
    print("\n各层参数梯度统计:")
    for i, (grad, var) in enumerate(zip(gradients, model.trainable_variables)):
        if grad is not None:
            grad_flat = tf.reshape(grad, [-1])
            print(f"\n  参数 {i+1} ({var.name}):")
            print(f"    形状: {grad.shape}")
            print(f"    梯度均值: {tf.reduce_mean(tf.abs(grad)).numpy():.8f}")
            print(f"    梯度标准差: {tf.math.reduce_std(grad).numpy():.8f}")
            print(f"    前5个梯度值: {grad_flat[:5].numpy().round(8)}")

# 交叉熵梯度示例
demonstrate_gradient_update(model_ce, 'categorical_crossentropy', x_train, y_train)

# MSE梯度示例
demonstrate_gradient_update(model_mse, 'mse', x_train, y_train)

## 2.6 可视化对比结果

In [ ]:
# 绘制对比图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epoch_range = range(1, 11)

# 训练损失对比
axes[0, 0].plot(epoch_range, history_ce.history['loss'], 'b-', label='交叉熵', linewidth=2)
axes[0, 0].plot(epoch_range, history_mse.history['loss'], 'r-', label='均方误差', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('训练损失对比')
axes[0, 0].legend()
axes[0, 0].grid(True)

# 验证损失对比
axes[0, 1].plot(epoch_range, history_ce.history['val_loss'], 'b-', label='交叉熵', linewidth=2)
axes[0, 1].plot(epoch_range, history_mse.history['val_loss'], 'r-', label='均方误差', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('验证损失对比')
axes[0, 1].legend()
axes[0, 1].grid(True)

# 训练准确率对比
axes[1, 0].plot(epoch_range, history_ce.history['accuracy'], 'b-', label='交叉熵', linewidth=2)
axes[1, 0].plot(epoch_range, history_mse.history['accuracy'], 'r-', label='均方误差', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('训练准确率对比')
axes[1, 0].legend()
axes[1, 0].grid(True)

# 验证准确率对比
axes[1, 1].plot(epoch_range, history_ce.history['val_accuracy'], 'b-', label='交叉熵', linewidth=2)
axes[1, 1].plot(epoch_range, history_mse.history['val_accuracy'], 'r-', label='均方误差', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('验证准确率对比')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('task2_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n对比图已保存到 task2_loss_comparison.png")

---

# 实验总结

## 任务一总结

1. **数据预处理**：MNIST数据集包含60000张训练图像和10000张测试图像，每张图像为28x28像素的灰度图。预处理包括归一化（将像素值缩放到[0,1]）、展平（将2D图像转换为1D向量）和one-hot编码（将标签转换为10维向量）。

2. **模型结构**：使用Keras Sequential API构建了一个包含两个隐藏层的全连接神经网络，使用ReLU激活函数和Dropout正则化防止过拟合，输出层使用Softmax激活函数进行多分类。

3. **训练过程**：使用Adam优化器和交叉熵损失函数，经过10个epoch的训练，模型在测试集上达到了较高的准确率。

4. **前向传播与反向传播**：前向传播逐层计算输出，反向传播使用链式法则计算梯度并更新权重。

## 任务二总结

1. **交叉熵损失**：专门为分类任务设计，与Softmax配合效果好，收敛速度快，最终准确率高。

2. **均方误差损失**： originally 用于回归任务，在分类任务中表现较差，收敛速度慢，可能遇到梯度消失问题。

3. **结论**：对于多分类任务，交叉熵损失是更好的选择。

## 实验体会

通过本次实训，深入理解了神经网络的前向传播和反向传播机制，掌握了Keras高层API的使用方法，并验证了不同损失函数对分类任务性能的影响。